# ATML PA3 — Task 1 finish: OPLoRA-16 training (GPU 1) + evaluation of all 7 conditions (GPU 0)

GPU T4 x2, Internet on. Attach `atml-pa3-code` **and** the outputs of training sessions 1 and 2
(Add Input → Your Work → the training notebook → pick each version). Then *Save & Run All (Commit)*.
GPU 0 evaluates every condition whose export exists (lm-eval HF backend + 24 qualitative prompts);
OPLoRA is evaluated as soon as its training on GPU 1 finishes. Rerunning with this notebook's own
output attached skips anything already finished.

In [ ]:
# --- locate code (Kaggle dataset "atml-pa3-code" containing atml-pa3-code.zip, or Colab Drive) ---
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content/drive/MyDrive/atml-pa3"
if not ON_KAGGLE:
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
REPO = "/tmp/ATML-PA3"           # code lives outside the saved output
zips = glob.glob("/kaggle/input/**/atml-pa3-code.zip", recursive=True) + glob.glob(f"{WORK}/atml-pa3-code.zip")
# Kaggle usually unpacks an uploaded zip, so also accept an already-extracted copy
dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/task1/train.py", recursive=True)]
shutil.rmtree(REPO, ignore_errors=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(REPO)
else:
    assert dirs, "upload outputs/atml-pa3-code.zip as a Kaggle dataset (or to Drive/atml-pa3/)"
    shutil.copytree(dirs[0], REPO)
print("code from", zips[:1] or dirs[:1])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "hf_xet"], check=True)
os.chdir(REPO)
os.makedirs(f"{WORK}/outputs", exist_ok=True)
if not os.path.exists("outputs"):
    os.symlink(f"{WORK}/outputs", "outputs")
import torch; print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# restore training exports/logs (not resume states) and earlier eval results from attached outputs
for prev in sorted(glob.glob("/kaggle/input/**/outputs/task1/*", recursive=True)):
    dst = f"outputs/task1/{os.path.basename(prev)}"
    if os.path.isdir(prev) and not os.path.exists(dst):
        print("restoring", prev); shutil.copytree(prev, dst, ignore=shutil.ignore_patterns("resume.tmp"))
for prev in sorted(glob.glob("/kaggle/input/**/outputs/results_task1/*", recursive=True)):
    dst = f"outputs/results_task1/{os.path.basename(prev)}"
    if os.path.isdir(prev) and not os.path.exists(dst):
        print("restoring", prev); shutil.copytree(prev, dst)
print("attached inputs:", os.listdir("/kaggle/input") if ON_KAGGLE else "-")
need = {"full": "export_u1250", "lora4": "export_u1250", "l2sp": "export_u1250", "lora16": "export_u1250"}
missing = [v for v, e in need.items() if not os.path.exists(f"outputs/task1/{v}/{e}/export_info.json")]
assert not missing, (f"training exports not found for {missing}: attach the session-1 and session-2 versions "
                     "of the training notebook (Add Input -> Your Work -> Notebooks) and rerun")
print("found all training exports")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lm_eval==0.4.13", "langdetect", "immutabledict"], check=True)
!ls outputs/task1/*

In [ ]:
import json, time
T_START, BUDGET_H = time.time(), 10.5
EXPORT = {"full_u625": "full/export_u625", "full": "full/export_u1250", "l2sp": "l2sp/export_u1250",
          "lora4": "lora4/export_u1250", "lora16": "lora16/export_u1250", "oplora16": "oplora16/export_u1250"}
CONDITIONS = ["baseline", "full_u625", "full", "l2sp", "lora4", "lora16", "oplora16"]
RES, MODELS = "outputs/results_task1", "/kaggle/temp/eval_models" if ON_KAGGLE else "/content/eval_models"
ready = lambda c: c == "baseline" or os.path.exists(f"outputs/task1/{EXPORT[c]}/export_info.json")
evaluated = lambda c: os.path.exists(f"{RES}/{c}/qualitative.json")

train = None
if not ready("oplora16"):
    os.makedirs("outputs/task1/oplora16", exist_ok=True)
    env1 = dict(os.environ, CUDA_VISIBLE_DEVICES="1" if torch.cuda.device_count() > 1 else "0",
                PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    train = subprocess.Popen([sys.executable, "-m", "task1.train", "--variant", "oplora16",
                              "--out", "outputs/task1/oplora16", "--time_limit_h", "10.3"],
                             stdout=open("outputs/task1/oplora16/stdout.log", "a"), stderr=subprocess.STDOUT, env=env1)
    print("started oplora16 training")

env0 = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
while True:
    todo = [c for c in CONDITIONS if ready(c) and not evaluated(c)]
    if not todo:
        if train is not None and train.poll() is None:
            print(time.strftime("%H:%M"), "waiting for oplora16:",
                  open("outputs/task1/oplora16/stdout.log").read().splitlines()[-1:], flush=True)
            time.sleep(300); continue
        break
    c = todo[0]
    t0 = time.time()
    left = BUDGET_H * 3600 - (time.time() - T_START)
    if left < 600:
        print("time budget reached; attach this output and rerun to continue", flush=True)
        break
    subprocess.run([sys.executable, "-m", "task1.prepare_eval", "--out", MODELS, "--only", c], check=True, env=env0)
    try:   # stop cleanly before Kaggle's 12 h limit so everything finished so far is saved
        subprocess.run([sys.executable, "-m", "task1.evaluate", "--backend", "hf", "--models", MODELS,
                        "--out", RES, "--conditions", c], check=True, env=env0, timeout=left)
    except subprocess.TimeoutExpired:
        print("time budget reached during", c, flush=True)
        break
    shutil.rmtree(f"{MODELS}/{c}", ignore_errors=True)
    print(time.strftime("%H:%M"), "evaluated", c, f"in {(time.time() - t0) / 60:.0f} min", flush=True)
# earlier conditions ran GSM8K at batch size 1 (lm-eval "auto" fallback), later ones at 32:
# rerun the baseline at 32 once to measure whether batching changes the scores
CHECK = f"{RES}/_batchcheck"
left = BUDGET_H * 3600 - (time.time() - T_START)
if all(evaluated(c) for c in CONDITIONS) and left > 1800 and not glob.glob(f"{CHECK}/baseline/gsm8k/**/results_*.json", recursive=True):
    subprocess.run([sys.executable, "-m", "task1.prepare_eval", "--out", MODELS, "--only", "baseline"], check=True, env=env0)
    try:
        subprocess.run([sys.executable, "-m", "task1.evaluate", "--backend", "hf", "--models", MODELS, "--out", CHECK,
                        "--conditions", "baseline", "--skip", "lik", "qual"], check=True, env=env0, timeout=left)
    except subprocess.TimeoutExpired:
        print("time budget reached during batch check", flush=True)
if os.path.exists(f"{MODELS}/merge_checks.json"):
    os.makedirs(RES, exist_ok=True)
    old = json.load(open(f"{RES}/merge_checks.json")) if os.path.exists(f"{RES}/merge_checks.json") else {}
    json.dump({**old, **json.load(open(f"{MODELS}/merge_checks.json"))}, open(f"{RES}/merge_checks.json", "w"), indent=2)
print("missing:", [c for c in CONDITIONS if not evaluated(c)])

In [ ]:
import json
for c in CONDITIONS:
    for f in glob.glob(f"{RES}/{c}/**/results_*.json", recursive=True):
        r = json.load(open(f))["results"]
        print(c, {t: {k: round(v, 4) for k, v in m.items() if isinstance(v, float) and "stderr" not in k} for t, m in r.items()})
!du -sh outputs/*